In [1]:
!pip install numpy torch tqdm semanticscholar pymupdf chromadb transformers networkx lxml accelerate bitsandbytes pyvis

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 131.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 125.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import os
import json
import numpy as np
import random
from collections import defaultdict
import torch
from tqdm import tqdm
import chromadb
import networkx as nx
from semanticscholar import SemanticScholar
import requests
import fitz
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
import time
from tenacity import RetryError
from pyvis.network import Network


In [3]:
# setup
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PATH_DATA = "./data"

PATH_JSON = f"{PATH_DATA}/papers.json"
PATH_TEXT = f"{PATH_DATA}/text"
PATH_PDF = f"{PATH_DATA}/pdf"

PATH_KG = f"{PATH_DATA}/kg.graphml"

PATH_CHROMA_DB = f"{PATH_DATA}/chroma"

PATH_TEACHER_INPUTS = "./data/teacher_inputs.jsonl"
PATH_STUDENT_INPUTS = "./data/student_inputs.jsonl"

os.makedirs(PATH_TEXT, exist_ok=True)
os.makedirs(PATH_PDF, exist_ok=True)

# embedding model for scientific papers
specter2 = "allenai/specter2_base"
tokenizer = AutoTokenizer.from_pretrained(specter2)
model_embedding = AutoModel.from_pretrained(specter2)
model_embedding.to(DEVICE)
model_embedding.eval()

# API for scientific papers
sch = SemanticScholar(api_key="s2k-15WdAzOKzJZ5WQHQrQXxsWJA61XJBTATl2kVYv5Q")

# vector database
chroma_client = chromadb.PersistentClient(path=PATH_CHROMA_DB)
collection = chroma_client.get_or_create_collection(name="papers")

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/228k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/717k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
# get paper data if it already exists
papers = []

if os.path.exists(PATH_JSON):
    with open(PATH_JSON, "r", encoding="utf8") as f:
        papers = json.load(f)

# ids of papers
papers_id = {x["paper_id"]: x for x in papers}

# get KG if it already exists
if os.path.exists(PATH_KG):
    G = nx.read_graphml(PATH_KG)
else:
    G = nx.MultiDiGraph()

In [5]:
# functions for downloading and storing data

# embed text into vector
@torch.no_grad()
def embed_text(text: str):

    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    outputs = model_embedding(**inputs)

    # mean pooling
    embeddings = outputs.last_hidden_state.mean(dim=1)
    embeddings = embeddings.cpu().numpy()[0]

    return embeddings


# store text in txt file
def store_text(paper_id, text):
    path = os.path.join(
        PATH_TEXT,
        f"{paper_id}.txt"
    )

    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


# get path of pdf file
def get_pdf(paper_id, url):
    if not url:
        return None

    try:
        response = requests.get(
            url, timeout=30
        )

        if response.status_code != 200:
            return None

        pdf_path = os.path.join(PATH_PDF, f"{paper_id}.pdf")

        with open(pdf_path, "wb") as f:
            f.write(response.content)

        return pdf_path

    except Exception as e:
        print("pdf error: ", e)
        return None


# get text from pdf file
def get_text(pdf_path):
    if not pdf_path:
        return ""

    try:
        pdf = fitz.open(pdf_path)
        text = ""

        for page in pdf:
            text = text + page.get_text()
        pdf.close()

        return text


    except Exception as e:
        print("text error", e)
        return ""

In [6]:
# download data and store it

# RUN ONLY IF NO DATA DOWNLOADED YET
# RESTART KERNEL AND DELETE data FOLDER BEFORE RUNNING
# CODE MAY TAKE VERY LONG BECAUSE OF LIMITED API AVAILABILITY

# topics to look for via API
TOPICS = [
    "artificial intelligence",
    # "machine learning",
    "deep learning",
    # "reinforcement learning",
    "large language models",
    "retrieval augmented generation",
    #"transformers",
]

# minimum citations needed for paper
CITATIONS_MIN = 20

# one paper might be chosen from multiple topics
papers_done = set()

# store which topics retrieved which papers
paper_topics = defaultdict(set)

# final papers list
papers = []

for idx_topic, topic in enumerate(TOPICS):

    time.sleep(10)
    print(idx_topic)

    # fix code failing because of limited API availability
    attempts = 0
    while attempts < 5:
        try:
            results = sch.search_paper(
                topic,
                limit=15,
                fields=[
                    "paperId", "title", "abstract", "year", "authors",
                    "citationCount", "referenceCount", "references.paperId",
                    "venue", "fieldsOfStudy", "openAccessPdf", "externalIds", "url"
                ]
            )
            break
        except Exception as e:
            if "500" in str(e) or "RetryError" in type(e).__name__:
                attempts += 1
                print(f"attempt {attempts}/5, waiting 30s: ", e)
                time.sleep(30)
            else:
                print("ERROR: ", e)
                raise e
    else:
        print(f"skipping topic '{topic}' after 5 failed attempts")
        continue

    # store data from papers
    for idx_paper, paper in enumerate(tqdm(results.items)):
        print(idx_topic, paper)

        try:
            paper_id = paper.paperId

            if not paper_id:
                continue

            title = paper.title or ""
            abstract = paper.abstract or ""
            year = paper.year or 0
            venue = paper.venue or ""
            count_citation = paper.citationCount or 0
            count_reference = paper.referenceCount or 0
            fields_of_study = paper.fieldsOfStudy or []
            url = None

            if hasattr(paper, "openAccessPdf") and paper.openAccessPdf:
                if isinstance(paper.openAccessPdf, dict):
                    url = paper.openAccessPdf.get("url")

            # skip papers that do not fulfill requirements
            # if not url:
            #     continue
            if count_citation < CITATIONS_MIN:
                continue
            if not title.strip():
                continue
            if not abstract.strip():
                continue


            if paper_id in papers_done:
                paper_topics[paper_id].add(topic)
                continue

            papers_done.add(paper_id)
            paper_topics[paper_id].add(topic)

            # store authors
            authors = []
            if paper.authors:
                for x in paper.authors:
                    authors.append({
                        "authorId": x.authorId,
                        "name": x.name
                    })

            # store citations (deactivated)
            citations = []
            if paper.citations:
                for x in paper.citations:
                    if x.paperId:
                        citations.append(x.paperId)


            # store references
            references = [
                x.paperId
                for x in paper.references
                if x.paperId
            ]


            # structure chromadb embeddings
            embedding_text = f"""
            TITLE: {title}

            ABSTRACT: {abstract}
            """

            # embed text
            embedding = embed_text(embedding_text)

            # get pdf path
            pdf_path = get_pdf(paper_id=paper_id, url=url)

            # get paper text
            paper_text = get_text(pdf_path=pdf_path)

            # skip paper if it is corrupted (less than 500 words)
            # if len(paper_text.split()) < 500:
            #     print("skipped ", title)
            #     continue


            # store paper
            text = f"""
            TITLE: {title}

            ABSTRACT: {abstract}

            VENUE: {venue}

            YEAR: {year}

            AUTHORS: {json.dumps(authors, indent=2)}

            FIELDS: {json.dumps(fields_of_study, indent=2)}

            REFERENCES: {json.dumps(references, indent=2)}

            CITATIONS: {json.dumps(citations, indent=2)}

            PDF_URL: {url}

            TEXT: {paper_text}
            """
            store_text(paper_id, text)


            # store in chromadb
            collection.add(
                ids=[paper_id],
                embeddings=[embedding.tolist()],
                documents=[embedding_text],
                metadatas=[{
                    "title": title,
                    "year": year,
                    "venue": venue,
                    "count_citation": count_citation,
                    "count_reference": count_reference,
                    "topics": ",".join(paper_topics[paper_id]),
                    "text": f"{paper_id}.txt",
                    "url": url
                }]
            )


            # add node to KG
            G.add_node(
                paper_id,
                type="paper",
                label=title,
                title=title,
                year=year,
                venue=venue,
                count_citation=count_citation,
                topics=";".join(paper_topics[paper_id])
            )

            # author relations
            for x in authors:
                author_id = x["authorId"]
                author_name = x["name"]

                if not author_id:
                    continue

                G.add_node(
                    author_id,
                    type="author",
                    label=author_name,
                    name=author_name
                )

                G.add_edge(
                    author_id,
                    paper_id,
                    relation="WROTE"
                )

            # citation relations
            for x in references:

                G.add_edge(
                    paper_id,
                    x,
                    relation="CITES"
                )


            # field_of_study relations
            for x in fields_of_study:
                field_node = f"field:{x}"

                G.add_node(
                    field_node,
                    type="field_of_study",
                    label=str(x),
                    name=str(x)
                )

                G.add_edge(
                    paper_id,
                    field_node,
                    relation="HAS_FIELD"
                )



            # metadata
            papers.append({
                "paper_id": paper_id,
                "title": title,
                "abstract": abstract,
                "year": year,
                "venue": venue,
                "authors": authors,
                "citations": citations,
                "references": references,
                "fields_of_study": fields_of_study,
                "count_citation": count_citation,
                "count_reference": count_reference,
                "topics": list(paper_topics[paper_id]),
                "file_name": f"{paper_id}.txt",
                "topic_original": topic
            })

        except Exception as e:
            print(e)


    # store papers
    with open(PATH_JSON, "w", encoding="utf-8") as f:
        json.dump(
            papers,
            f,
            indent=2,
            ensure_ascii=False
        )

    # write to KG
    nx.write_graphml(G, PATH_KG)

# ids of papers
papers_id = {
    x["paper_id"]: x
    for x in papers
}



# remove references without paper from KG
papers_present = set(papers_id.keys())
nodes_to_remove = []

for node, data in G.nodes(data=True):
    if data.get("type") == "author":
        continue
    if data.get("type") == "field_of_study":
        continue

    if node not in papers_present:
        nodes_to_remove.append(node)

G.remove_nodes_from(nodes_to_remove)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

0


  0%|          | 0/15 [00:00<?, ?it/s]

0 {'paperId': '530a059cb48477ad1e3d4f8f4b153274c8997332', 'externalIds': {'MAG': '2997428643', 'ArXiv': '1910.10045', 'DBLP': 'journals/corr/abs-1910-10045', 'DOI': '10.1016/j.inffus.2019.12.012', 'CorpusId': 204824113}, 'url': 'https://www.semanticscholar.org/paper/530a059cb48477ad1e3d4f8f4b153274c8997332', 'title': 'Explainable Artificial Intelligence (XAI): Concepts, Taxonomies, Opportunities and Challenges toward Responsible AI', 'venue': 'Information Fusion', 'year': 2019, 'referenceCount': 427, 'citationCount': 8804, 'openAccessPdf': {'url': 'https://bird.bcamath.org/bitstream/20.500.11824/1166/1/24.%20main.pdf', 'status': 'GREEN', 'license': 'CCBYNCSA', 'disclaimer': 'Notice: Paper or abstract available at https://arxiv.org/abs/1910.10045, which is subject to the license by the author or copyright owner provided with this content. Please go to the source to verify the license and copyright information for your use.'}, 'fieldsOfStudy': ['Computer Science'], 'authors': [{'authorId

  7%|▋         | 1/15 [00:03<00:51,  3.67s/it]

...

3 {'paperId': '659bf9ce7175e1ec266ff54359e2bd76e0b7ff31', 'externalIds': {'DBLP': 'conf/nips/LewisPPPKGKLYR020', 'MAG': '3027879771', 'ArXiv': '2005.11401', 'CorpusId': 218869575}, 'url': 'https://www.semanticscholar.org/paper/659bf9ce7175e1ec266ff54359e2bd76e0b7ff31', 'title': 'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks', 'venue': 'Neural Information Processing Systems', 'year': 2020, 'referenceCount': 67, 'citationCount': 15292, 'openAccessPdf': {'url': '', 'status': None, 'license': None, 'disclaimer': 'Notice: Paper or abstract available at https://arxiv.org/abs/2005.11401, which is subject to the license by the author or copyright owner provided with this content. Please go to the source to verify the license and copyright information for your use.'}, 'fieldsOfStudy': ['Computer Science'], 'authors': [{'authorId': '145222654', 'name': 'Patrick Lewis'}, {'authorId': '3439053', 'name': 'Ethan Perez'}, {'authorId': '1716179427', 'name': 'Aleksandara Piktus'}, {'

 67%|██████▋   | 10/15 [00:00<00:00, 28.23it/s]

3 {'paperId': 'ba7952e7c4fb891c36980ca19f94251257da6eb7', 'externalIds': {'DBLP': 'journals/corr/abs-2501-09136', 'ArXiv': '2501.09136', 'DOI': '10.48550/arXiv.2501.09136', 'CorpusId': 275570331}, 'url': 'https://www.semanticscholar.org/paper/ba7952e7c4fb891c36980ca19f94251257da6eb7', 'title': 'Agentic Retrieval-Augmented Generation: A Survey on Agentic RAG', 'venue': 'arXiv.org', 'year': 2025, 'referenceCount': 71, 'citationCount': 332, 'openAccessPdf': {'url': '', 'status': None, 'license': None, 'disclaimer': 'Notice: Paper or abstract available at https://arxiv.org/abs/2501.09136, which is subject to the license by the author or copyright owner provided with this content. Please go to the source to verify the license and copyright information for your use.'}, 'fieldsOfStudy': ['Computer Science'], 'authors': [{'authorId': '2109435730', 'name': 'Aditi Singh'}, {'authorId': '2284079039', 'name': 'Abul Ehtesham'}, {'authorId': '2310654424', 'name': 'Saket Kumar'}, {'authorId': '241873

100%|██████████| 15/15 [00:00<00:00, 25.04it/s]

3 {'paperId': '4b8588b56d5f0ffab3f19f7e90a9416f247b6f05', 'externalIds': {'DBLP': 'journals/corr/abs-2502-01549', 'ArXiv': '2502.01549', 'DOI': '10.1145/3770854.3783944', 'CorpusId': 276107937}, 'url': 'https://www.semanticscholar.org/paper/4b8588b56d5f0ffab3f19f7e90a9416f247b6f05', 'title': 'VideoRAG: Retrieval-Augmented Generation with Extreme Long-Context Videos', 'venue': 'Knowledge Discovery and Data Mining', 'year': 2025, 'referenceCount': 38, 'citationCount': 63, 'openAccessPdf': {'url': '', 'status': None, 'license': None, 'disclaimer': 'Notice: Paper or abstract available at https://arxiv.org/abs/2502.01549, which is subject to the license by the author or copyright owner provided with this content. Please go to the source to verify the license and copyright information for your use.'}, 'fieldsOfStudy': ['Computer Science'], 'authors': [{'authorId': '2163180478', 'name': 'Xubin Ren'}, {'authorId': '2343886330', 'name': 'Lingrui Xu'}, {'authorId': '2287796033', 'name': 'Long Xi

In [7]:
# visualize KG
def visualize_kg(G, node_id, radius=2):

    sub = nx.ego_graph(G, node_id, radius=radius)

    net = Network(height="750px", width="100%", notebook=True)

    color_map = {
        "paper": "orange",
        "author": "blue",
        "field_of_study": "black"
    }

    for n, attributes in sub.nodes(data=True):
        ntype = attributes.get("type", "unknown")
        color = color_map.get(ntype, "gray")
        label = attributes.get("label", str(n))

        net.add_node(n, label=label, color=color)

    for u, v, attributes in sub.edges(data=True):
        net.add_edge(u, v, label=attributes.get("relation", ""))

    net.show("kg.html")

node_id = papers[1]["paper_id"]

visualize_kg(G, node_id, radius=1)


kg.html


In [9]:
# functions to get relevant papers

# get vector of a specific paper
def get_embedding_id(paper_id):
    result = collection.get(
        ids=[paper_id],
        include=["embeddings"]
    )

    return result["embeddings"][0]

# get similar papers from vector db
def get_similar_papers(paper_id, k=5):
    embedding = get_embedding_id(paper_id)

    result = collection.query(
        query_embeddings=[embedding],
        n_results=k+1
    )

    ids = result["ids"][0]

    return [
        x for x in ids
        if x != paper_id
    ][:k]

# get information from the KG of a specific paper
def get_kg(paper_id):
    information_kg = {
        "authors": [],
        "references": [],
        "fields_of_study": []
    }

    if not G.has_node(paper_id):
        return information_kg

    for source, target, data in G.edges(data=True):
        relation = data.get("relation")

        if target == paper_id and relation == "WROTE":
            name = G.nodes[source].get("name", source)
            information_kg["authors"].append(name)

        elif source == paper_id and relation == "CITES":
            information_kg["references"].append(target)

        elif source == paper_id and relation == "HAS_FIELD":
            name = target.replace("field:", "")
            information_kg["fields_of_study"].append(name)

    return information_kg

# gets relevant information from KG and vector db
def get_relevant_papers(paper_id, limit=5):
    kg = get_kg(paper_id)

    return {
        "similar_papers": get_similar_papers(paper_id, k=limit),
        "cites": kg["references"][:limit],
        "authors": kg["authors"],
        "fields": kg["fields_of_study"],
    }

In [10]:
# functions to create prompts

# get paper data
def get_paper_summary(paper_id):
    if paper_id not in papers_id:
        return None

    x = papers_id[paper_id]

    return {
        "title": x["title"],
        "abstract": x["abstract"],
        "year": x["year"],
        "venue": x["venue"],
        "count_citation": x["count_citation"]
    }

# structure paper data
def make_json(paper):
    return {
        # "paper_id": paper["paper_id"],
        "title": paper["title"],
        "abstract": paper["abstract"],
        "year": paper["year"],
        "venue": paper["venue"],
        "fields_of_study": paper["fields_of_study"],
        "count_citation": paper["count_citation"]
    }

# make input for the teacher model
def make_teacher_input(paper_id):
    paper = papers_id[paper_id]
    papers_relevant = get_relevant_papers(paper_id)

    similar_papers = [
        get_paper_summary(x)
        for x in papers_relevant["similar_papers"] if x in papers_id
    ][:10]

    return {
        "paper": make_json(paper),
        "similar_papers": similar_papers,
        "kg_summary": {
            "authors": papers_relevant["authors"],
            "fields_of_study": papers_relevant["fields"],
            "citations": {
                "cites": [
                    {
                        "title": papers_id[x]["title"],
                        "year": papers_id[x]["year"],
                        "authors": [a["name"] for a in papers_id[x]["authors"]],
                        "venue": papers_id[x]["venue"]
                    }
                    for x in papers_relevant["cites"]
                    if x in papers_id
                ]
            }
        }

    }

# add questions to teacher input
def make_teacher_prompt(teacher_input, question):
    return f"""
    You get scientific paper data and a question regarding how they relate. Use only the provided data.

    Question: {question}

    - Paper: {teacher_input["paper"]}
    - Similar papers: {teacher_input["similar_papers"]}
    - Citations: {teacher_input["kg_summary"]["citations"]}
    - Authors: {teacher_input["kg_summary"]["authors"]}
    - Fields: {teacher_input["kg_summary"]["fields_of_study"]}

    """

In [11]:
# store teacher inputs
with open(PATH_TEACHER_INPUTS, "w", encoding="utf-8") as f:
    for x in papers:
        id = x["paper_id"]
        input = make_teacher_input(id)

        f.write(
            json.dumps(
                input, ensure_ascii=False
            ) + "\n"
        )

In [12]:
# questions to learn different tasks
QUESTIONS = [
    "How do {topic} papers relate?",
    "How did research evolve regarding {topic}?",
]

# teacher model, using 7b-version to achieve more meaningful teaching effect
model_teacher_name = "Qwen/Qwen2.5-7B-Instruct"
#model_teacher_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer_teacher = AutoTokenizer.from_pretrained(model_teacher_name)

# gpu
model_teacher = AutoModelForCausalLM.from_pretrained(
     model_teacher_name,
     device_map="auto",
     quantization_config=bnb_config
)

# cpu
#model_teacher = AutoModelForCausalLM.from_pretrained(
#    model_teacher_name,
#    device_map="cpu"
#)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [13]:
# functions to create the final dataset to fine tune the student model

# query teacher model
def call_teacher(prompt, max_new_tokens=300):
    inputs = tokenizer_teacher(prompt, return_tensors="pt").to(model_teacher.device)

    outputs = model_teacher.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.3,
        top_p=0.9,
        do_sample=True)

    # only taking tokens generated after the prompt ends
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer_teacher.decode(generated, skip_special_tokens=True)

# get topic that was used to find the paper via API
def get_topic_original(paper):
    topic = paper.get("topic_original")
    return topic

# create final dataset for the student model
def get_student_input(paper_id):

    teacher_input = make_teacher_input(paper_id)
    topic = get_topic_original(papers_id[paper_id])

    question = random.choice(QUESTIONS).format(topic=topic)

    prompt = make_teacher_prompt(teacher_input, question)

    response = call_teacher(prompt)

    return {
        "question": question,
        "input": teacher_input,
        "output": response
    }


In [14]:
# store input for student model
data_student = []

with open(PATH_STUDENT_INPUTS, "w", encoding="utf-8") as f:
    for idx, x in enumerate(papers):
        print(idx)
        try:
            paper_id = x["paper_id"]
            data = get_student_input(paper_id)
            data_student.append(data)
            f.write(json.dumps(data, ensure_ascii=False) + "\n")
            f.flush()
        except Exception as e:
            print(paper_id, e)


0


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51


In [15]:
model_student_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer_student = AutoTokenizer.from_pretrained(model_student_name)
model_student = AutoModelForCausalLM.from_pretrained(model_student_name, device_map="cuda")
model_student.train()

data_train = [json.loads(l) for l in open(PATH_STUDENT_INPUTS, encoding="utf-8")]
optimizer = torch.optim.AdamW(model_student.parameters(), lr=5e-5)

for x in data_train:
    prompt = make_teacher_prompt(x["input"], x["question"])
    text = prompt + x["output"]
    inputs = tokenizer_student(text, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model_student.device) for k, v in inputs.items()}
    outputs = model_student(**inputs, labels=inputs["input_ids"])
    outputs.loss.backward()
    optimizer.step()
    optimizer.zero_grad()

model_student.save_pretrained("./data/student_model")
tokenizer_student.save_pretrained("./data/student_model")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./data/student_model/tokenizer_config.json',
 './data/student_model/chat_template.jinja',
 './data/student_model/tokenizer.json')

## TF-IDF Baseline

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus_ids = [x["paper_id"] for x in papers]
corpus_text = [f"{x['title']} {x['abstract']}" for x in papers]

# ignoring stopwords, maxdf handles shared keywords
vectorizer_tfidf = TfidfVectorizer(stop_words="english", max_df=0.7)
matrix_tfidf = vectorizer_tfidf.fit_transform(corpus_text)

def get_similar_papers_tfidf(paper_id, k=5):
    idx = corpus_ids.index(paper_id)
    scores = cosine_similarity(matrix_tfidf[idx], matrix_tfidf).flatten()
    ranked = scores.argsort()[::-1]
    result = []
    for i in ranked:
        if corpus_ids[i] != paper_id:
            result.append(corpus_ids[i])
        if len(result) == k:
            break
    return result

def get_shared_keywords(paper_id, other_id, top_n=5):
    idx_a = corpus_ids.index(paper_id)
    idx_b = corpus_ids.index(other_id)
    vec_a = matrix_tfidf[idx_a].toarray().flatten()
    vec_b = matrix_tfidf[idx_b].toarray().flatten()
    features = vectorizer_tfidf.get_feature_names_out()
    shared_scores = vec_a * vec_b
    ranked = shared_scores.argsort()[::-1]
    return [features[i] for i in ranked[:top_n] if shared_scores[i] > 0]

def make_tfidf_explanation(paper_id, other_id):
    paper = papers_id[paper_id]
    other = papers_id[other_id]
    keywords = get_shared_keywords(paper_id, other_id)
    cites = other_id in paper["references"]
    cited_by = paper_id in other["references"]
    relation = "cites" if cites else "is cited by" if cited_by else "no direct citation link"
    return f'"{paper["title"]}" and "{other["title"]}" share the terms: {", ".join(keywords)}. Relation: {relation}.'

def run_baseline_tfidf(paper_id, k=5):
    similar = get_similar_papers_tfidf(paper_id, k=k)
    return {
        "paper_id": paper_id,
        "similar_papers": similar,
        "explanations": [make_tfidf_explanation(paper_id, x) for x in similar]
    }

In [17]:
print(f"papers loaded: {len(papers)}")
print(f"tfidf matrix shape: {matrix_tfidf.shape}")

test_id = corpus_ids[0]
test_result = run_baseline_tfidf(test_id, k=3)
print(f"test paper: {papers_id[test_id]['title']}")
print(f"similar found: {len(test_result['similar_papers'])}")
for exp in test_result["explanations"]:
    print(exp)

papers loaded: 52
tfidf matrix shape: (52, 2199)
test paper: Explainable Artificial Intelligence (XAI): Concepts, Taxonomies, Opportunities and Challenges toward Responsible AI
similar found: 3
"Explainable Artificial Intelligence (XAI): Concepts, Taxonomies, Opportunities and Challenges toward Responsible AI" and "Peeking Inside the Black-Box: A Survey on Explainable Artificial Intelligence (XAI)" share the terms: xai, ai, explainable, intelligence, artificial. Relation: cites.
"Explainable Artificial Intelligence (XAI): Concepts, Taxonomies, Opportunities and Challenges toward Responsible AI" and "ARTIFICIAL INTELLIGENCE IN EDUCATION: CHALLENGES AND OPPORTUNITIES FOR SUSTAINABLE DEVELOPMENT" share the terms: ai, opportunities, intelligence, artificial, responsible. Relation: no direct citation link.
"Explainable Artificial Intelligence (XAI): Concepts, Taxonomies, Opportunities and Challenges toward Responsible AI" and "Revolutionizing healthcare: the role of artificial intelligence 

## SPECTER-2 with RAG

In [18]:
def make_rag_input(paper_id, k=5):
    paper = papers_id[paper_id]
    similar_ids = get_similar_papers(paper_id, k=k)
    similar_papers = [get_paper_summary(x) for x in similar_ids if x in papers_id]
    return {"paper": make_json(paper), "similar_papers": similar_papers}

def make_rag_prompt(rag_input, question):
    return f"""
    You get scientific paper data and a question regarding how they relate. Use only the provided data.

    Question: {question}

    - Paper: {rag_input["paper"]}
    - Similar papers: {rag_input["similar_papers"]}
    """

def run_baseline_rag(paper_id, k=5):
    rag_input = make_rag_input(paper_id, k=k)
    topic = get_topic_original(papers_id[paper_id])
    question = random.choice(QUESTIONS).format(topic=topic)
    prompt = make_rag_prompt(rag_input, question)
    response = call_teacher(prompt)
    return {"paper_id": paper_id, "question": question, "input": rag_input, "output": response}

In [19]:
print(f"chroma collection count: {collection.count()}")

test_id = papers[0]["paper_id"]
test_result = run_baseline_rag(test_id, k=3)
print(f"test paper: {papers_id[test_id]['title']}")
print(f"question: {test_result['question']}")
print(f"similar papers used: {len(test_result['input']['similar_papers'])}")
print(f"output length (chars): {len(test_result['output'])}")
print(test_result["output"][:500])

chroma collection count: 52
test paper: Explainable Artificial Intelligence (XAI): Concepts, Taxonomies, Opportunities and Challenges toward Responsible AI
question: How do artificial intelligence papers relate?
similar papers used: 3
output length (chars): 706
 - Metadata: {'doi': '10.1016/j.inffus.2019.02.003', 'pdf_url': 'https://www.sciencedirect.com/science/article/pii/S1574013719300241'}
    - Metadata: {'doi': '10.1109/ACCESS.2018.2853564', 'pdf_url': 'https://ieeexplore.ieee.org/document/8486297'}
    - Metadata: {'doi': '10.1016/j.ijinfomgt.2019.02.001', 'pdf_url': 'https://www.sciencedirect.com/science/article/pii/S0268401219300117'}
    - Metadata: {'doi': '10.1007/s11704-023-00833-8', 'pdf_url': 'https://link.springer.com/content/pdf/10.100


In [20]:
def call_student(prompt, max_new_tokens=300):
    inputs = tokenizer_student(prompt, return_tensors="pt").to(model_student.device)
    outputs = model_student.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.3, top_p=0.9, do_sample=True)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer_student.decode(generated, skip_special_tokens=True)

def run_full_pipeline(paper_id, k=5):
    teacher_input = make_teacher_input(paper_id)
    topic = get_topic_original(papers_id[paper_id])
    question = random.choice(QUESTIONS).format(topic=topic)
    prompt = make_teacher_prompt(teacher_input, question)
    response = call_student(prompt)
    return {"paper_id": paper_id, "question": question, "input": teacher_input, "output": response}

In [21]:
import re

def judge_score(paper_id, output_text):
    paper = papers_id[paper_id]
    prompt = f'Rate 1-10 how well this text explains how the paper "{paper["title"]}" relates to other papers. Reply with only a number.\n\nText: {output_text}'
    response = call_teacher(prompt, max_new_tokens=10)
    match = re.search(r"\b(10|[1-9])\b", response)
    return int(match.group(1)) if match else None

def hallucination_rate(output_text, teacher_input):
    allowed = {p["title"] for p in teacher_input["similar_papers"]}
    allowed |= {c["title"] for c in teacher_input["kg_summary"]["citations"]["cites"]}
    allowed.add(teacher_input["paper"]["title"])
    mentioned = [p["title"] for p in papers if p["title"] in output_text]
    if not mentioned:
        return 0
    hallucinated = [t for t in mentioned if t not in allowed]
    return len(hallucinated) / len(mentioned)

In [22]:
results_compare = []

for x in papers[:15]:
    paper_id = x["paper_id"]
    teacher_input = make_teacher_input(paper_id)

    r_base = run_baseline_tfidf(paper_id)
    r_rag = run_baseline_rag(paper_id)
    r_full = run_full_pipeline(paper_id)

    for name, r in [("tfidf", r_base), ("rag", r_rag), ("full", r_full)]:
        output_text = " ".join(r["explanations"]) if name == "tfidf" else r["output"]
        results_compare.append({
            "experiment": name,
            "judge_score": judge_score(paper_id, output_text),
            "hallucination_rate": hallucination_rate(output_text, teacher_input)
        })

for name in ["tfidf", "rag", "full"]:
    scores = [r["judge_score"] for r in results_compare if r["experiment"] == name and r["judge_score"] is not None]
    halluc = [r["hallucination_rate"] for r in results_compare if r["experiment"] == name]
    print(name, "judge_score:", sum(scores)/len(scores) if scores else None, "hallucination_rate:", sum(halluc)/len(halluc) if halluc else None)

tfidf judge_score: 4.866666666666666 hallucination_rate: 0.3888888888888889
rag judge_score: 5.0 hallucination_rate: 0.0
full judge_score: 5.666666666666667 hallucination_rate: 0.6666666666666666
